# 03 — Spark execution lab

This notebook uses small synthetic datasets to demonstrate how Spark executes work. The experiments are deliberately separate from airline business analysis so plans, stages, and shuffles are easier to inspect.

Run it in the project's Spark environment. The notebook inherits the configured Spark Standalone master and workers; it does not install PySpark, start services, or hardcode a master URL.

## 1. Session setup

A small shuffle-partition count keeps physical plans and task counts readable. Executor placement and exact task timing still depend on the external cluster.

In [1]:
import time

from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("airline-03-spark-execution-lab").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
original_shuffle_partitions = spark.conf.get("spark.sql.shuffle.partitions")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print(f"Spark application: {spark.sparkContext.appName}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")
print(f"SQL shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")

Spark application: airline-03-spark-execution-lab
Default parallelism: 2
SQL shuffle partitions: 4


### Spark UI orientation

Open the Spark UI supplied by the project environment while running each action. Use **Jobs** to see actions becoming jobs, **Stages** to find shuffle boundaries, **SQL/DataFrame** to inspect query plans, **Executors** to see worker activity, and **Storage** to inspect cached data. A completed action may disappear from the active list, so also check completed jobs and stages.

## 2. RDD versus DataFrame

An RDD is a distributed collection of language objects manipulated with functional transformations. A DataFrame adds named columns and a schema, allowing Spark SQL's optimizer to reason about the computation. Both examples calculate squares of even integers.

In [2]:
numbers_rdd = spark.sparkContext.parallelize(range(1, 11), 2)
even_squares_rdd = numbers_rdd.filter(lambda value: value % 2 == 0).map(lambda value: (value, value * value))
print("RDD partitions:", even_squares_rdd.getNumPartitions())
print("RDD result:", even_squares_rdd.take(10))

RDD partitions: 2


RDD result: [(2, 4), (4, 16), (6, 36), (8, 64), (10, 100)]


In [3]:
numbers_df = spark.range(1, 11, numPartitions=2)
even_squares_df = (
    numbers_df.filter(F.col("id") % 2 == 0)
    .select(F.col("id").alias("value"), (F.col("id") * F.col("id")).alias("square"))
)
even_squares_df.show()

+-----+------+
|value|square|
+-----+------+
|    2|     4|
|    4|    16|
|    6|    36|
|    8|    64|
|   10|   100|
+-----+------+



Both APIs distribute work, but the DataFrame exposes column types and expressions to Catalyst. In the Spark UI, the RDD action appears as an RDD job, while `show()` also has an entry in the SQL/DataFrame view.

## 3. Lazy evaluation and transformations versus actions

`filter`, `withColumn`, and `select` describe transformations but do not immediately process every row. An action such as `count`, `show`, or `write` asks Spark to execute the accumulated plan. `explain()` prints the plan without materializing the result.

In [4]:
lazy_df = (
    spark.range(0, 100_000, numPartitions=4)
    .filter(F.col("id") % 3 == 0)
    .withColumn("squared", F.col("id") * F.col("id"))
)
print("Transformations are defined; no result action has run yet.")
lazy_df.explain(mode="simple")

lazy_count = lazy_df.count()
print(f"Action result: {lazy_count:,} rows")

Transformations are defined; no result action has run yet.
== Physical Plan ==
*(1) Project [id#17L, (id#17L * id#17L) AS squared#21L]
+- *(1) Filter ((id#17L % 3) = 0)
   +- *(1) Range (0, 100000, step=1, splits=4)




Action result: 33,334 rows


Inspect the Jobs page immediately before and after `count()`. Defining the transformations should not create a data-processing job; the action should. Spark may run supporting jobs depending on runtime features, so use the SQL description and job group to identify the relevant work.

## 4. Narrow versus wide transformations and shuffle

A narrow transformation can compute each output partition from one input partition, as with `map` or `filter`. A wide transformation needs records from multiple input partitions to meet, as with `groupBy`, `reduceByKey`, many joins, or `repartition`. Moving those records across executors is a **shuffle**.

In [5]:
def rdd_lineage(rdd):
    lineage = rdd.toDebugString()
    return lineage.decode("utf-8") if isinstance(lineage, bytes) else lineage

pairs_rdd = spark.sparkContext.parallelize(range(20), 4).map(lambda value: (value % 3, value))
narrow_rdd = pairs_rdd.filter(lambda pair: pair[1] % 2 == 0).mapValues(lambda value: value * 10)
wide_rdd = narrow_rdd.reduceByKey(lambda left, right: left + right)

print("Narrow lineage:")
print(rdd_lineage(narrow_rdd))
print("\nWide lineage; look for ShuffledRDD:")
print(rdd_lineage(wide_rdd))
print("Result:", wide_rdd.take(10))

Narrow lineage:
(4) PythonRDD[19] at RDD at PythonRDD.scala:53 []
 |  ParallelCollectionRDD[14] at readRDDFromFile at PythonRDD.scala:289 []

Wide lineage; look for ShuffledRDD:
(4) PythonRDD[20] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[18] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[17] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(4) PairwiseRDD[16] at reduceByKey at /tmp/ipykernel_5240/2065398972.py:7 []
    |  PythonRDD[15] at reduceByKey at /tmp/ipykernel_5240/2065398972.py:7 []
    |  ParallelCollectionRDD[14] at readRDDFromFile at PythonRDD.scala:289 []


Result: [(0, 360), (1, 300), (2, 240)]


In [6]:
events = spark.range(0, 100_000, numPartitions=8).withColumn("bucket", F.col("id") % 10)
narrow_plan = events.filter(F.col("id") % 2 == 0).select("bucket", "id")
wide_plan = narrow_plan.groupBy("bucket").agg(F.count("*").alias("rows"), F.sum("id").alias("id_sum"))

print("Narrow DataFrame physical plan:")
narrow_plan.explain(mode="formatted")
print("Wide DataFrame physical plan; look for Exchange:")
wide_plan.explain(mode="formatted")
wide_plan.orderBy("bucket").show()

Narrow DataFrame physical plan:
== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#31L]
Arguments: Range (0, 100000, step=1, splits=Some(8))

(2) Filter [codegen id : 1]
Input [1]: [id#31L]
Condition : ((id#31L % 2) = 0)

(3) Project [codegen id : 1]
Output [2]: [(id#31L % 10) AS bucket#33L, id#31L]
Input [1]: [id#31L]


Wide DataFrame physical plan; look for Exchange:
== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Project (3)
            +- Filter (2)
               +- Range (1)


(1) Range
Output [1]: [id#31L]
Arguments: Range (0, 100000, step=1, splits=Some(8))

(2) Filter
Input [1]: [id#31L]
Condition : ((id#31L % 2) = 0)

(3) Project
Output [2]: [(id#31L % 10) AS bucket#33L, id#31L]
Input [1]: [id#31L]

(4) HashAggregate
Input [2]: [bucket#33L, id#31L]
Keys [1]: [bucket#33L]
Functions [2]: [partial_count(1), partial_sum(id#31L)]
Aggregate Attribut

+------+-----+---------+
|bucket| rows|   id_sum|
+------+-----+---------+
|     0|10000|499950000|
|     2|10000|499970000|
|     4|10000|499990000|
|     6|10000|500010000|
|     8|10000|500030000|
+------+-----+---------+



In an RDD lineage, `ShuffledRDD` marks the wide dependency. In a DataFrame physical plan, `Exchange` commonly marks data redistribution. In the Stage detail page, compare shuffle read/write metrics and the number of tasks on each side of that boundary.

## 5. Jobs, stages, and tasks

An action starts a job. Spark divides the job into stages at shuffle boundaries, then launches tasks—normally one task per partition processed in that stage. Adaptive Query Execution can change the final number of shuffle partitions, so inspect the executed plan and UI rather than assuming a fixed count.

In [7]:
spark.sparkContext.setJobGroup("execution-lab-stages", "Jobs, stages, tasks experiment")
stage_demo = (
    spark.range(0, 200_000, numPartitions=8)
    .withColumn("group_id", F.col("id") % 20)
    .groupBy("group_id")
    .agg(F.avg("id").alias("average_id"))
)
stage_demo.orderBy("group_id").show(20)
spark.sparkContext.setLocalProperty("spark.jobGroup.id", None)
spark.sparkContext.setLocalProperty("spark.job.description", None)

+--------+----------+
|group_id|average_id|
+--------+----------+
|       0|   99990.0|
|       1|   99991.0|
|       2|   99992.0|
|       3|   99993.0|
|       4|   99994.0|
|       5|   99995.0|
|       6|   99996.0|
|       7|   99997.0|
|       8|   99998.0|
|       9|   99999.0|
|      10|  100000.0|
|      11|  100001.0|
|      12|  100002.0|
|      13|  100003.0|
|      14|  100004.0|
|      15|  100005.0|
|      16|  100006.0|
|      17|  100007.0|
|      18|  100008.0|
|      19|  100009.0|
+--------+----------+



Find the job named **Jobs, stages, tasks experiment**. The aggregation introduces a shuffle and therefore a stage boundary. On each stage page, compare input partitions, task count, shuffle bytes, executor assignment, and task-duration variation. The final ordering can introduce another exchange.

## 6. `repartition()` versus `coalesce()`

`repartition()` uses a shuffle to redistribute data and can increase or decrease partition count. `coalesce()` normally combines existing partitions without a full shuffle and is best suited to reducing partitions when balance is not critical.

In [8]:
partition_base = spark.range(0, 1_000, numPartitions=8)
repartitioned = partition_base.repartition(4)
coalesced = partition_base.coalesce(4)

print("Original partitions:", partition_base.rdd.getNumPartitions())
print("After repartition(4):", repartitioned.rdd.getNumPartitions())
print("After coalesce(4):", coalesced.rdd.getNumPartitions())
print("\nrepartition physical plan; look for Exchange:")
repartitioned.explain(mode="formatted")
print("coalesce physical plan; look for Coalesce without an Exchange:")
coalesced.explain(mode="formatted")
repartitioned.count(), coalesced.count()

Original partitions: 8


After repartition(4): 4
After coalesce(4): 4

repartition physical plan; look for Exchange:
== Physical Plan ==
AdaptiveSparkPlan (5)
+- == Final Plan ==
   ShuffleQueryStage (3), Statistics(sizeInBytes=15.6 KiB, rowCount=1.00E+3)
   +- Exchange (2)
      +- * Range (1)
+- == Initial Plan ==
   Exchange (4)
   +- Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#91L]
Arguments: Range (0, 1000, step=1, splits=Some(8))

(2) Exchange
Input [1]: [id#91L]
Arguments: RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=208]

(3) ShuffleQueryStage
Output [1]: [id#91L]
Arguments: 0

(4) Exchange
Input [1]: [id#91L]
Arguments: RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=203]

(5) AdaptiveSparkPlan
Output [1]: [id#91L]
Arguments: isFinalPlan=true


coalesce physical plan; look for Coalesce without an Exchange:
== Physical Plan ==
Coalesce (2)
+- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#91L]
Arguments: Range (0, 1000, step=1, splits=Some(8))

(2) Coalesce
Input

(1000, 1000)

Compare plans and stage metrics. `repartition(4)` should include an exchange and shuffle activity; `coalesce(4)` should show a coalesce operator without a full redistribution. Partition balance may differ even when both finish with four partitions.

## 7. Join types

The left table contains flight-like rows and the right table contains airline names. Their unmatched keys make the differences between join types visible.

In [9]:
flight_rows = spark.createDataFrame(
    [(1, "AA"), (2, "DL"), (3, "XX"), (4, "AA")],
    ["flight_id", "airline"],
).repartition(2)
airline_rows = spark.createDataFrame(
    [("AA", "Example Air A"), ("DL", "Example Air D"), ("UA", "Example Air U")],
    ["airline", "airline_name"],
)

print("Left table:")
flight_rows.orderBy("flight_id").show()
print("Right table:")
airline_rows.orderBy("airline").show()

Left table:


+---------+-------+
|flight_id|airline|
+---------+-------+
|        1|     AA|
|        2|     DL|
|        3|     XX|
|        4|     AA|
+---------+-------+

Right table:


+-------+-------------+
|airline| airline_name|
+-------+-------------+
|     AA|Example Air A|
|     DL|Example Air D|
|     UA|Example Air U|
+-------+-------------+



In [10]:
print("Inner: matched rows only")
flight_rows.join(airline_rows, "airline", "inner").orderBy("flight_id").show()

print("Left: every left row, with null right values when unmatched")
flight_rows.join(airline_rows, "airline", "left").orderBy("flight_id").show()

print("Left semi: left rows whose key exists on the right; no right columns")
flight_rows.join(airline_rows, "airline", "left_semi").orderBy("flight_id").show()

print("Left anti: left rows whose key does not exist on the right")
flight_rows.join(airline_rows, "airline", "left_anti").orderBy("flight_id").show()

Inner: matched rows only


+-------+---------+-------------+
|airline|flight_id| airline_name|
+-------+---------+-------------+
|     AA|        1|Example Air A|
|     DL|        2|Example Air D|
|     AA|        4|Example Air A|
+-------+---------+-------------+

Left: every left row, with null right values when unmatched


+-------+---------+-------------+
|airline|flight_id| airline_name|
+-------+---------+-------------+
|     AA|        1|Example Air A|
|     DL|        2|Example Air D|
|     XX|        3|         NULL|
|     AA|        4|Example Air A|
+-------+---------+-------------+

Left semi: left rows whose key exists on the right; no right columns


+-------+---------+
|airline|flight_id|
+-------+---------+
|     AA|        1|
|     DL|        2|
|     AA|        4|
+-------+---------+

Left anti: left rows whose key does not exist on the right


+-------+---------+
|airline|flight_id|
+-------+---------+
|     XX|        3|
+-------+---------+



In [11]:
broadcast_join = flight_rows.join(F.broadcast(airline_rows), "airline", "left")
broadcast_join.orderBy("flight_id").show()
broadcast_join.explain(mode="formatted")

+-------+---------+-------------+
|airline|flight_id| airline_name|
+-------+---------+-------------+
|     AA|        1|Example Air A|
|     DL|        2|Example Air D|
|     XX|        3|         NULL|
|     AA|        4|Example Air A|
+-------+---------+-------------+

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Project (7)
   +- BroadcastHashJoin LeftOuter BuildRight (6)
      :- Exchange (2)
      :  +- Scan ExistingRDD (1)
      +- BroadcastExchange (5)
         +- Filter (4)
            +- Scan ExistingRDD (3)


(1) Scan ExistingRDD
Output [2]: [flight_id#109L, airline#110]
Arguments: [flight_id#109L, airline#110], MapPartitionsRDD[85] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Exchange
Input [2]: [flight_id#109L, airline#110]
Arguments: RoundRobinPartitioning(2), REPARTITION_BY_NUM, [plan_id=1145]

(3) Scan ExistingRDD
Output [2]: [airline#113, airline_name#114]
Arguments: [airline#113, airline_name#114], MapPartition

Use semi joins for existence filtering and anti joins for missing-key checks. For the broadcast example, look for `BroadcastHashJoin` and `BroadcastExchange` in the plan, then inspect the SQL and stage pages to see that the small right side is distributed without a large two-sided shuffle.

## 8. Caching and persistence

Caching is useful when the same expensive result is reused. The first action materializes the cache; later actions can reuse stored partitions. Timing is shown only as an observation—cluster load can make a cached action slower in a particular run.

In [12]:
expensive = (
    spark.range(0, 500_000, numPartitions=8)
    .withColumn("bucket", F.col("id") % 100)
    .withColumn("score", F.sqrt(F.col("id") + 1) * F.log1p(F.col("id")))
    .groupBy("bucket").agg(F.avg("score").alias("average_score"))
)

cached = expensive.persist(StorageLevel.MEMORY_AND_DISK)
first_start = time.perf_counter()
first_count = cached.count()
first_seconds = time.perf_counter() - first_start

second_start = time.perf_counter()
second_sum = cached.agg(F.sum("average_score")).first()[0]
second_seconds = time.perf_counter() - second_start

print(f"Cached rows: {first_count}")
print(f"First action (materializes cache): {first_seconds:.3f} seconds")
print(f"Second action (can reuse cache): {second_seconds:.3f} seconds")
print(f"Second action result: {second_sum:.3f}")
print(f"Storage level: {cached.storageLevel}")

Cached rows: 100
First action (materializes cache): 2.104 seconds
Second action (can reuse cache): 0.663 seconds
Second action result: 587168.102
Storage level: Disk Memory Serialized 1x Replicated


After the first action, inspect **Storage** for cached partitions and size. In the second SQL plan, look for `InMemoryTableScan`. Compare stage input and shuffle metrics, not just elapsed time. Always release data that is no longer reused.

In [13]:
cached.unpersist(blocking=True)
print("Cached experiment data released.")

Cached experiment data released.


## 9. Reading `explain()` and physical plans

A formatted plan separates the physical operator tree from operator details. Read from the data source upward and look for filters, projections, exchanges, join strategies, aggregations, sorts, and adaptive-plan markers.

In [14]:
plan_demo = (
    events.filter(F.col("id") % 5 == 0)
    .groupBy("bucket")
    .agg(F.count("*").alias("rows"), F.max("id").alias("max_id"))
    .orderBy(F.desc("rows"))
)
plan_demo.explain(mode="formatted")
plan_demo.show()

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Range (1)


(1) Range
Output [1]: [id#31L]
Arguments: Range (0, 100000, step=1, splits=Some(8))

(2) Filter
Input [1]: [id#31L]
Condition : ((id#31L % 5) = 0)

(3) Project
Output [2]: [id#31L, (id#31L % 10) AS bucket#33L]
Input [1]: [id#31L]

(4) HashAggregate
Input [2]: [id#31L, bucket#33L]
Keys [1]: [bucket#33L]
Functions [2]: [partial_count(1), partial_max(id#31L)]
Aggregate Attributes [2]: [count#363L, max#364L]
Results [3]: [bucket#33L, count#365L, max#366L]

(5) Exchange
Input [3]: [bucket#33L, count#365L, max#366L]
Arguments: hashpartitioning(bucket#33L, 4), ENSURE_REQUIREMENTS, [plan_id=1304]

(6) HashAggregate
Input [3]: [bucket#33L, count#365L, max#366L]
Keys [1]: [bucket#33L]
Functions [2]: [count(1), max(id#31L)]
Aggregate Attributes 

+------+-----+------+
|bucket| rows|max_id|
+------+-----+------+
|     5|10000| 99995|
|     0|10000| 99990|
+------+-----+------+



Expected landmarks include a filter near the scan, partial and final aggregates, an exchange for grouping, and another exchange or sort for global ordering. With Adaptive Query Execution enabled, the final executed plan may differ from the initial plan; the SQL page shows both.

In [15]:
spark.conf.set("spark.sql.shuffle.partitions", original_shuffle_partitions)
print("Original SQL shuffle-partition setting restored.")

Original SQL shuffle-partition setting restored.


## 10. Lab summary

Spark builds lazy plans, starts work at actions, divides jobs at shuffle boundaries, and runs tasks over partitions. DataFrames add optimization and readable physical plans; repartitioning, joins, and aggregation shape network movement; caching trades storage for reuse. The Spark UI connects these concepts to actual execution.

In [16]:
spark.stop()
print("Execution lab complete; Spark session stopped.")

Execution lab complete; Spark session stopped.
